In [ ]:
import numpy as np

import matplotlib.pyplot as plt
from pathlib import Path
import re

In [ ]:
paths = ["output/resnet50/"]

In [ ]:
def reduce(x: np.ndarray) -> np.ndarray:
    if len(x.shape) == 1:
        return x
    return x.mean(1)

In [ ]:
curves: dict[str, np.ndarray] = {}
emprts: dict[str, np.ndarray] = {}
smprts: dict[str, np.ndarray] = {}

names = []

for path in paths:
    for p in Path(path).iterdir():
        re_name = re.match(r"^(?P<TYPE>.+)_(?P<NAME>.+).csv$", p.name)

        if re_name is None: continue

        name = re_name.group("NAME")
        data = np.loadtxt(p, delimiter=",")

        if name not in names:
            names.append(name)

        print(name, data.shape)

        match re_name.group("TYPE").lower():
            case "perturbation_curve": curves[name] = data
            case "emprt": emprts[name] = reduce(data)
            case "smprt": smprts[name] = reduce(data)
            case x: raise ValueError(f"No such thing '{x}'.")

In [ ]:
_names = names
# _names = ["IG", "GIG", "IIG-2"]
# _names = ["IG", "IIG-2"]

for name in _names:
    curve = curves[name]
    plt.plot(curve.mean(0), label=name)

plt.title("Perturbation curves")

plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# area under curve
aucs = {}
for name, curve in curves.items():
    aucs[name] = curve.mean(0).sum()

In [ ]:
for name, auc in sorted(aucs.items(), key=lambda x: x[1]):
    print(f"{name:5s}: {auc:.2f}")

In [ ]:
plt.boxplot(list(emprts.values()))
plt.xticks(range(1, len(emprts)+1), list(emprts.keys()))

plt.title("eMPRT")

plt.tight_layout()
plt.show()

In [ ]:
plt.boxplot(list(smprts.values()))
plt.xticks(range(1, len(smprts)+1), list(smprts.keys()))

plt.title("sMPRT")

plt.tight_layout()
plt.show()